# Final Policy Compliance Pipeline

Embedding model: BGE-large `BAAI/bge-large-en-v1.5`, Re-ranker model: Qwen3-Reranker-4B, LLM model: Mistral-7B-Instruct-v0.3


---
## Section 1 — Imports & Config


In [ ]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
import torch
import transformers

from collections import defaultdict, Counter
from dataclasses import dataclass, asdict

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Paths ────────────────────────────────────────────────────
JCI_PDF     = '/Users/siri/Desktop/sghpolicyreview/data/ESS Standards_7th Ed .pdf'
DB_SAVE_DIR = 'saved_dbs'

# ── Embedding model ──────────────────────────────────────────
EMBEDDING_MODEL_NAME   = 'BGE-large'
EMBEDDING_MODEL_STRING = 'BAAI/bge-large-en-v1.5'

# ── Reranker ─────────────────────────────────────────────────
QWEN_MODEL_ID = 'Qwen/Qwen3-Reranker-4B'

# ── LLM ──────────────────────────────────
LLM_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

# ── Retrieval settings ───────────────────────────────────────
PER_DB_K   = 8    # chunks fetched per chapter DB in vector search
FINAL_K    = 10   # chunks kept after global re-sort and dedup
MIN_CHUNKS = 7    # minimum chunks needed; Tier 1 retrieval supplements if below this

# ── LLM settings ───────────────────────────
LLM_MAX_TOKENS = 512

# ── JCI chapter codes ────────────────────────────────────────
VALID_CODES = {
    'APR', 'IPSG', 'ACC', 'PCC', 'AOP', 'COP', 'ASC', 'MMU',
    'QPS', 'PCI',  'GLD', 'FMS', 'SQE', 'MOI', 'MPE', 'HRP',
}
END_OF_BOOK_MARKERS = ['glossary', 'index', 'appendix', 'references']

# ── Parser config ────────────────────────────────────────────
VALID_CHAPTERS = VALID_CODES


TASK_KEYWORDS = {
    'compliance_check': [
        'comply', 'complies', 'compliance', 'aligned', 'aligns',
        'meets', 'meet', 'satisfies', 'consistent with', 'in line with',
        'conform', 'gap', 'gaps', 'missing', 'is this', 'does this',
        'adequate', 'fulfil',
        # Added based on queries
        'satisfy', 'sufficient', 'fully meet', 'fully satisfy',
        'fully align', 'is sufficient', 'does this satisfy',
        'does this meet', 'does this align', 'does this fully',
        'is this sufficient',
    ],
    'comparison': [
        'compare', 'comparison', 'versus', 'vs', 'differ',
        'difference', 'how does', 'contrast', 'against',
        # Added based on queries
        'how does this compare', 'compare to', 'compared to',
        'compared with',
    ],
    'clarification': [
        'what does', 'what is', 'what are', 'explain', 'define',
        'describe', 'clarify', 'what requirement', 'what criteria',
        'what should', 'how should',
        # Added based on queries
        'what additional', 'what is the basis', 'what basis',
        'what elements', 'what documentation',
    ],
}
print('Section 1 complete — config loaded.')

Section 1 complete — config loaded.


---
## Section 2 — Load, Chunk & Build FAISS DB: JCI ESS
- Loads the JCI ESS PDF and groups pages by chapter
- Splits each chapter into standard-level chunks with `standard_code` metadata
- Builds one FAISS DB per chapter under `saved_dbs/BGE-large/<chapter>/`
- `chapter_chunks` dict is also kept in memory for Tier 1 direct lookup
- **Run once** — skips if DBs already exist

In [ ]:
def load_pdf(path: str):
    pages = PyPDFLoader(path).load()
    print(f'Loaded {len(pages)} pages from JCI PDF.')
    return pages


def detect_chapter(text: str):
    """Returns JCI chapter code if this page is a chapter title page."""
    for line in text.split('\n'):
        m = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line.strip())
        if m and m.group(1) in VALID_CODES:
            return m.group(1)
    return None


def is_end_of_book(text: str) -> bool:
    return any(marker in text.strip()[:200].lower() for marker in END_OF_BOOK_MARKERS)


def group_pages_by_chapter(pages) -> dict:
    chapter_pages = defaultdict(list)
    current = 'UNKNOWN'
    for page in pages:
        detected = detect_chapter(page.page_content)
        if detected:
            current = detected
        chapter_pages[current].append(page)

    if 'HRP' in chapter_pages:
        before = len(chapter_pages['HRP'])
        chapter_pages['HRP'] = [
            p for p in chapter_pages['HRP'] if not is_end_of_book(p.page_content)
        ]
        print(f'HRP: removed {before - len(chapter_pages["HRP"])} end-of-book pages.')

    print('\n=== Chapter page counts ===')
    for ch, pgs in chapter_pages.items():
        print(f'  {ch}: {len(pgs)} pages')
    return chapter_pages


def build_chapter_chunks(chapter_pages: dict, chunk_size: int = 1500,
                          chunk_overlap: int = 150) -> dict:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=['\n\n', '\n', '. ', ' '],
    )
    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        full_text = '\n'.join(p.page_content for p in pg_list)
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text)
        full_text = re.sub(r'\n{3,}', '\n\n', full_text)

        toc = re.search(
            r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements',
            full_text, re.IGNORECASE
        )
        if toc:
            full_text = full_text[toc.start():]

        pattern = (
            rf'(?:\n|^)(?:Standard|Requirement)\s+'
            rf'({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?(?:\s*\.\s*\d+)?)\b'
        )
        splits = re.split(pattern, full_text)

        docs = []
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={'chapter': chapter, 'standard_code': 'INTRO'},
            ))

        for i in range(1, len(splits), 2):
            if i + 1 >= len(splits):
                break
            code    = splits[i].strip().replace(' ', '')
            content = f'{code}\n{splits[i + 1]}'
            if len(content.strip()) < 100:
                continue
            if len(content) <= chunk_size:
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={'chapter': chapter, 'standard_code': code},
                ))
            else:
                for j, sub in enumerate(splitter.split_text(content)):
                    if len(sub.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=sub.strip(),
                        metadata={'chapter': chapter, 'standard_code': code, 'chunk_index': j},
                    ))

        chapter_chunks[chapter] = docs
        unique = len(set(d.metadata['standard_code'] for d in docs))
        print(f'  {chapter}: {len(docs)} chunks across {unique} standards')

    return chapter_chunks


pages          = load_pdf(JCI_PDF)
chapter_pages  = group_pages_by_chapter(pages)
chapter_chunks = build_chapter_chunks(chapter_pages)

# ── Build and save chapter-level FAISS DBs ───────────────────────
model_dir = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME)

if os.path.exists(model_dir) and any(
    os.path.isdir(os.path.join(model_dir, ch)) for ch in VALID_CODES
):
    print(f'[{EMBEDDING_MODEL_NAME}] JCI DBs already exist — skipping build.')
else:
    print(f'[{EMBEDDING_MODEL_NAME}] Building JCI chapter DBs...')
    os.makedirs(model_dir, exist_ok=True)
    jci_build_embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_STRING,
        encode_kwargs={'normalize_embeddings': True},
    )
    for chapter, docs in chapter_chunks.items():
        if docs:
            db = FAISS.from_documents(docs, jci_build_embeddings)
            db.save_local(os.path.join(model_dir, chapter))
            print(f'  Saved: {chapter}')
    print(f'[{EMBEDDING_MODEL_NAME}] JCI DBs done.')


print('Section 2 complete.')

Loaded 406 pages from JCI PDF.
HRP: removed 13 end-of-book pages.

=== Chapter page counts ===
  UNKNOWN: 2 pages
  FMS: 42 pages
  APR: 18 pages
  ACC: 22 pages
  PCC: 21 pages
  AOP: 30 pages
  COP: 32 pages
  ASC: 16 pages
  QPS: 15 pages
  PCI: 27 pages
  GLD: 32 pages
  SQE: 28 pages
  MOI: 25 pages
  MPE: 13 pages
  HRP: 32 pages
  IPSG: 16 pages
  MMU: 22 pages
  UNKNOWN: 1 chunks across 1 standards
  FMS: 79 chunks across 27 standards
  APR: 22 chunks across 12 standards
  ACC: 50 chunks across 19 standards
  PCC: 53 chunks across 22 standards
  AOP: 71 chunks across 36 standards
  COP: 80 chunks across 29 standards
  ASC: 32 chunks across 16 standards
  QPS: 37 chunks across 12 standards
  PCI: 66 chunks across 22 standards
  GLD: 92 chunks across 34 standards
  SQE: 79 chunks across 26 standards
  MOI: 59 chunks across 17 standards
  MPE: 16 chunks across 8 standards
  HRP: 80 chunks across 10 standards
  IPSG: 48 chunks across 14 standards
  MMU: 50 chunks across 19 standard

In [10]:
# ── Diagnostic: print all ACC standard codes, chunk counts and content ──
print('=== ACC standard codes in chapter_chunks ===\n')
acc_codes = {}
for doc in chapter_chunks.get('ACC', []):
    code = doc.metadata.get('standard_code')
    acc_codes[code] = acc_codes.get(code, 0) + 1

print('--- Chunk counts per standard ---')
for code, count in sorted(acc_codes.items()):
    print(f'  "{code}": {count} chunks')

print('\n--- Full chunk content ---')
for doc in chapter_chunks.get('ACC', []):
    code = doc.metadata.get('standard_code')
    chunk_idx = doc.metadata.get('chunk_index', 0)
    print(f'\n{"="*60}')
    print(f'Standard: {code} | Chunk index: {chunk_idx}')
    print(f'{"="*60}')
    print(doc.page_content.strip())

=== ACC standard codes in chapter_chunks ===

--- Chunk counts per standard ---
  "ACC.1": 4 chunks
  "ACC.1.1": 2 chunks
  "ACC.2": 3 chunks
  "ACC.2.1": 1 chunks
  "ACC.2.2": 1 chunks
  "ACC.2.3": 3 chunks
  "ACC.3": 4 chunks
  "ACC.3.1": 2 chunks
  "ACC.4": 1 chunks
  "ACC.4.1": 5 chunks
  "ACC.4.2": 2 chunks
  "ACC.4.2.1": 1 chunks
  "ACC.4.3": 2 chunks
  "ACC.4.4": 1 chunks
  "ACC.4.4.1": 4 chunks
  "ACC.5": 3 chunks
  "ACC.5.1": 2 chunks
  "ACC.6": 8 chunks
  "INTRO": 1 chunks

--- Full chunk content ---

Standard: INTRO | Chunk index: 0
Standards, Intents, and Measurable Elements.
Screening for Admission to the Hospital
ACC.1 Patients who may be admitted to the hospital or who seek outpatient services are screened to 
identify if their health care needs match the hospital’s mission and resources, and those with 
emergent, urgent, or immediate needs are given priority for assessment and treatment. 𝖯
ACC.1.1 The hospital considers the clinical needs of patients and informs patient

---
## Section 3 — Load BGE-large Embeddings & Retrieval Functions
- Loads BGE-large embedding model
- Loads all JCI chapter FAISS DBs into memory
- Defines `retrieve()` for vector search and `retrieve_by_standard_code()` for Tier 1 exact lookup
- Defines `route_and_retrieve()` with three-tier routing:
  - **Tier 1**: explicit standard code found → direct chunk lookup by metadata
  - **Tier 2**: chapter found → vector search within that chapter DB
  - **Tier 3**: nothing found → global vector search across all DBs

In [3]:
print(f'Loading embeddings model: {EMBEDDING_MODEL_STRING}')

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_STRING,
    encode_kwargs={'normalize_embeddings': True},
)

# ── Load JCI chapter DBs ─────────────────────────────────────
jci_dbs = {}
for chapter in chapter_chunks.keys():
    chapter_path = os.path.join(DB_SAVE_DIR, EMBEDDING_MODEL_NAME, chapter)
    if os.path.exists(chapter_path):
        jci_dbs[chapter] = FAISS.load_local(
            chapter_path, embeddings, allow_dangerous_deserialization=True,
        )
print(f'Loaded {len(jci_dbs)} JCI chapter DBs.')


def retrieve(query: str, dbs: dict) -> list:
    """
    Vector similarity search across given DBs.
    Returns list of (Document, l2_score) sorted by score ascending.
    Used by Tier 2 and Tier 3.
    """
    pool = []
    for chapter, db in dbs.items():
        for doc, score in db.similarity_search_with_score(query, k=PER_DB_K):
            doc.metadata['chapter'] = chapter
            pool.append((doc, score))

    pool.sort(key=lambda x: x[1])

    seen, unique = set(), []
    for doc, score in pool:
        key = (doc.page_content, doc.metadata.get('standard_code'))
        if key not in seen:
            unique.append((doc, score))
            seen.add(key)

    return unique[:FINAL_K]


def retrieve_by_standard_code(standard_codes: list, chapter_chunks: dict) -> list:
    """
    Tier 1: directly returns all chunks whose standard_code metadata
    matches any of the given codes. No vector search. Score = 0.0.
    
    e.g. ['ACC.5.1', 'AOP.1'] returns every chunk tagged with either code.
    """
    results = []
    for code in standard_codes:
        chapter = code.split('.')[0]   # 'ACC' from 'ACC.5.1'
        if chapter not in chapter_chunks:
            continue
        for doc in chapter_chunks[chapter]:
            if doc.metadata.get('standard_code') == code:
                results.append((doc, 0.0))  # 0.0 = exact match
    print(f'  Tier 1 direct lookup: {len(results)} chunks for {standard_codes}')
    return results


def route_and_retrieve(query: str, parsed, dbs: dict, chapter_chunks: dict, min_chunks: int = MIN_CHUNKS,) -> list:
    """
    Three-tier routing:

    Tier 1 — explicit standard code(s) found in query
             Directly looks up chunks by standard_code metadata.
             If fewer than min_chunks returned, supplements with
             chapter vector search to fill remaining slots.
             Exact-match chunks (score=0.0) always rank first.

    Tier 2 — chapter(s) found but no full standard code
             Vector search within each identified chapter DB.
             Multiple chapters are searched and merged.

    Tier 3 — nothing found
             Global vector search across all chapter DBs.
    """
    retrieval_query = query

    # ── Tier 1: exact standard code lookup ────────────────────────
    if parsed.all_standards:
        direct = retrieve_by_standard_code(parsed.all_standards, chapter_chunks)

        if len(direct) >= min_chunks:
            return direct

        # Not enough — supplement with chapter vector search
        print(f'  Tier 1 returned {len(direct)} chunks — supplementing with chapter search')
        seen = {(doc.page_content, doc.metadata.get('standard_code')) for doc, _ in direct}
        pool = list(direct)

        for chapter in parsed.all_chapters:
            if chapter in dbs:
                for doc, score in dbs[chapter].similarity_search_with_score(
                    retrieval_query, k=PER_DB_K
                ):
                    key = (doc.page_content, doc.metadata.get('standard_code'))
                    if key not in seen:
                        pool.append((doc, score))
                        seen.add(key)

        # Sort: exact matches (0.0) stay at top, similarity scores after
        pool.sort(key=lambda x: x[1])
        return pool[:FINAL_K]

    # ── Tier 2: chapter(s) known ─────────────────────────────────
    if parsed.all_chapters:
        pool = []
        seen = set()
        for chapter in parsed.all_chapters:
            if chapter in dbs:
                for doc, score in dbs[chapter].similarity_search_with_score(
                    retrieval_query, k=PER_DB_K
                ):
                    doc.metadata['chapter'] = chapter
                    key = (doc.page_content, doc.metadata.get('standard_code'))
                    if key not in seen:
                        pool.append((doc, score))
                        seen.add(key)
        if pool:
            pool.sort(key=lambda x: x[1])
            return pool[:FINAL_K]

    # ── Tier 3: global fallback ───────────────────────────────────
    print('  Tier 3 global search')
    return retrieve(retrieval_query, dbs)


print('Section 3 complete — embeddings and retrieval ready.')

Loading embeddings model: BAAI/bge-large-en-v1.5


/tmp/ipykernel_385/592114758.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4361.63it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 17 JCI chapter DBs.
Section 3 complete — embeddings and retrieval ready.


---
## Section 4 — Load Qwen Reranker
- Loads `Qwen3-Reranker-4B` as a causal LM
- Scores each (query, chunk) pair as yes/no relevance
- `norm_score` = log-softmax yes-probability → absolute 0–1


In [4]:
print(f'Loading Qwen reranker: {QWEN_MODEL_ID}')

qwen_r_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, padding_side='left')
qwen_r_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID, device_map='auto'
).eval()

qwen_r_true_id  = qwen_r_tok.convert_tokens_to_ids('yes')
qwen_r_false_id = qwen_r_tok.convert_tokens_to_ids('no')
qwen_r_maxlen   = 8192

_qwen_prefix = (
    '<|im_start|>system\n'
    'Judge whether the Document meets the requirements based on the '
    'Query and the Instruct provided. '
    'Note that the answer can only be "yes" or "no".'
    '<|im_end|>\n<|im_start|>user\n'
)
_qwen_suffix = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
qwen_r_prefix_toks = qwen_r_tok.encode(_qwen_prefix, add_special_tokens=False)
qwen_r_suffix_toks = qwen_r_tok.encode(_qwen_suffix, add_special_tokens=False)

_QWEN_INSTRUCTION = (
    'Given a hospital accreditation policy compliance query, retrieve the '
    'document chunk that contains the relevant JCI standard, intent, or '
    'measurable elements needed to assess compliance.'
)


def rerank_qwen(query: str, retrieved: list) -> list:
    """
    Returns list of (doc, norm_score) sorted by norm_score descending.
    norm_score = log_softmax yes-probability — absolute 0-1.
    Works identically whether chunks came from Tier 1, 2, or 3.
    """
    docs  = [doc for doc, _ in retrieved]
    pairs = [
        f'<Instruct>: {_QWEN_INSTRUCTION}\n'
        f'<Query>: {query}\n'
        f'<Document>: {doc.page_content[:400]}'
        for doc in docs
    ]

    inputs = qwen_r_tok(
        pairs, padding=False, truncation='longest_first',
        return_attention_mask=False,
        max_length=qwen_r_maxlen - len(qwen_r_prefix_toks) - len(qwen_r_suffix_toks),
    )
    for i, ele in enumerate(inputs['input_ids']):
        inputs['input_ids'][i] = qwen_r_prefix_toks + ele + qwen_r_suffix_toks
    inputs = qwen_r_tok.pad(
        inputs, padding=True, return_tensors='pt', max_length=qwen_r_maxlen
    )
    for key in inputs:
        inputs[key] = inputs[key].to(qwen_r_model.device)

    with torch.no_grad():
        logits = qwen_r_model(**inputs).logits[:, -1, :]
        tv     = logits[:, qwen_r_true_id]
        fv     = logits[:, qwen_r_false_id]
        scores = torch.nn.functional.log_softmax(
            torch.stack([fv, tv], dim=1), dim=1
        )[:, 1].exp().tolist()

    combined = [(doc, round(score, 4)) for doc, score in zip(docs, scores)]
    return sorted(combined, key=lambda x: x[1], reverse=True)


print('Section 4 complete — Qwen reranker loaded.')

Loading Qwen reranker: Qwen/Qwen3-Reranker-4B


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 452.33it/s]


Section 4 complete — Qwen reranker loaded.


---
## Section 5 — Query Parser
Updated to extract **multiple** standard codes and chapters from a single query.

| Field | What it extracts | Example |
|-------|------------------|---------|
| `task_type` | What the user wants | `compliance_check` |
| `target_policy` | Which hospital policy | `Transfer Policy` |
| `policy_clause` | Policy clause label | `5.1.4` |
| `reference_standard` | Primary JCI standard | `ACC.5.1` |
| `reference_chapter` | Primary JCI chapter | `ACC` |
| `all_standards` | **All** standards found | `['ACC.5.1', 'AOP.1']` |
| `all_chapters` | **All** chapters found | `['ACC', 'AOP']` |
| `measurable_element` | ME reference | `ME 3` |

**Routing priority:**
1. `all_standards` not empty → Tier 1 exact lookup
2. `all_chapters` not empty → Tier 2 chapter search
3. Nothing found → Tier 3 global

In [ ]:
@dataclass
class ParsedQuery:
    task_type:          str
    policy_clause:      str | None
    reference_standard: str | None   # primary (first found)
    reference_chapter:  str | None   # primary (first found)
    all_standards:      list          # all standard codes found
    all_chapters:       list          # all chapters found
    measurable_element: str | None
    retrieval_scope:    str           # 'targeted' / 'chapter' / 'global'
    routing_source:     str           # 'explicit' / 'inferred' / 'global'

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)


def extract_jci_standard(text: str) -> tuple[list, list]:
    """
    Extracts ALL JCI standard codes and chapters found in the query.
    Returns (list of standard codes, list of unique chapters).

    e.g. 'ACC.5.1 and AOP.1' -> (['ACC.5.1', 'AOP.1'], ['ACC', 'AOP'])
    e.g. 'ACC requirements'   -> ([], ['ACC'])
    e.g. nothing found        -> ([], [])
    """
    standards = []
    chapters  = []

    # First pass: find full standard codes e.g. ACC.5.1
    pattern = r'\b([A-Z]{2,4})\.(\d+(?:\.\d+){0,2})\b'
    for chapter, sub in re.findall(pattern, text):
        if chapter in VALID_CHAPTERS:
            std_code = f'{chapter}.{sub}'
            if std_code not in standards:
                standards.append(std_code)
            if chapter not in chapters:
                chapters.append(chapter)

    # Second pass: find standalone chapter codes not already captured
    for chapter in VALID_CHAPTERS:
        if chapter not in chapters:
            if re.search(rf'\b{chapter}\b', text):
                chapters.append(chapter)

    return standards, chapters


def extract_policy_clause(text: str) -> str | None:
    """
    Extracts hospital policy clause numbers e.g. 5.1.4.
    Excludes numbers that are part of a JCI standard code (e.g. the 5.1 in ACC.5.1).
    """
    for match in re.finditer(r'\b(\d+\.\d+(?:\.\d+)?)\b', text):
        start = match.start()
        preceding = text[:start].rstrip()
        if re.search(r'[A-Z]{2,4}\.$', preceding):
            continue   # part of a JCI code like ACC.5.1 — skip
        return match.group(1)
    return None


def extract_measurable_element(text: str) -> str | None:
    match = re.search(r'\bmeasurable\s+elements?\s*(\d+)\b', text, re.IGNORECASE)
    if match:
        return f'ME {match.group(1)}'
    match = re.search(r'\bme\s*(\d+)\b', text, re.IGNORECASE)
    if match:
        return f'ME {match.group(1)}'
    return None


def detect_task_type(text: str) -> str:
    text_lower = text.lower()
    for task_type, keywords in TASK_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return task_type
    return 'general_qa'


def parse_query(query: str, print_json: bool = True) -> ParsedQuery:
    """
    Rule-based parser — no LLM call.
    Extracts all standard codes and chapters found in the query.
    Primary fields use the first found value for backward compatibility.
    all_standards and all_chapters drive the three-tier routing.
    """
    task_type          = detect_task_type(query)
    policy_clause      = extract_policy_clause(query)
    measurable_element = extract_measurable_element(query)
    standards, chapters = extract_jci_standard(query)

    primary_standard = standards[0] if standards else None
    primary_chapter  = chapters[0]  if chapters  else None

    if primary_standard:
        scope, source = 'targeted', 'explicit'
    elif primary_chapter:
        scope, source = 'chapter', 'inferred'
    else:
        scope, source = 'global', 'global'

    result = ParsedQuery(
        task_type          = task_type,
        policy_clause      = policy_clause,
        reference_standard = primary_standard,
        reference_chapter  = primary_chapter,
        all_standards      = standards,
        all_chapters       = chapters,
        measurable_element = measurable_element,
        retrieval_scope    = scope,
        routing_source     = source,
    )

    if print_json:
        print(result.to_json())

    return result


print('Section 5 complete — parser ready.')

Parser sanity check:
----------------------------------------------------------------------
Query: Does clause 5.1 of our Transfer Policy comply with ACC.5.1?
{
  "task_type": "compliance_check",
  "policy_clause": "5.1",
  "reference_standard": "ACC.5.1",
  "reference_chapter": "ACC",
  "all_standards": [
    "ACC.5.1"
  ],
  "all_chapters": [
    "ACC"
  ],
  "measurable_element": null,
  "retrieval_scope": "targeted",
  "routing_source": "explicit"
}

Query: Compare our transfer process against ACC requirements
{
  "task_type": "comparison",
  "policy_clause": null,
  "reference_standard": null,
  "reference_chapter": "ACC",
  "all_standards": [],
  "all_chapters": [
    "ACC"
  ],
  "measurable_element": null,
  "retrieval_scope": "chapter",
  "routing_source": "inferred"
}

Query: What are the handover requirements for inter-hospital transfer?
{
  "task_type": "clarification",
  "policy_clause": null,
  "reference_standard": null,
  "reference_chapter": null,
  "all_standards": []

---
## Section 6 — LLM Generated Gold Standard Queries 
For ACC, PCC and COP chapters

In [ ]:
ACC_gold_standard = [
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 3.7 states "For Inter-hospital transfers, communication of the details of the transfer and the condition of the patient shall be done between doctors of at least Associate Consultant grade and above from both the referring and accepting team." Does this satisfy the requirements of JCI Standard ACC.5.1?',
        'expected_standard': 'ACC.5.1 ',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.1.4 states "The referring team should summarize the case history of patient and document it in SCM before transferring the patient to the other discipline. Information relayed includes health status of patient, summary of care provided, patients progress and the reason for transfer." Does this meet the requirements of JCI Standard ACC.3, Measurable Element 5?',
        'expected_standard': 'ACC.3',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.10 states "Critically ill patients requiring inter-hospital transfer should be accompanied by a doctor of at least MO or PGY2 Resident grade. The doctor in attendance should assume responsibility for en-route medical control during the transfer." How does this compare to the requirements of JCI Standard ACC.5, Measurable Element 3?',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'comparison',
    },
    {
        'query':             ': In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.11 states "Accompanying personnel shall be adequately equipped for any interventions required or anticipated for the patient. For example, staff capable of endotracheal intubation shall accompany an intubated patient or any patient with potential for airway compromise during an inter-hospital transfer." Does this satisfy JCI Standard ACC.5, Measurable Element 4?',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.7 states "The procedures and other interventions provided should be written in the patients discharge summary and given to the accepting staff taking over the care of the patient." Does this fully satisfy the documentation requirements of JCI Standard ACC.5.1, Measurable Elements 1 through 4?',
        'expected_standard': 'ACC.5.1',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 1.1 states "To provide guidelines for healthcare team to comply during intra-hospital and inter-hospital transfers of patients." Does this purpose statement fully align with the expectations set out in the JCI ACC chapter?',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.13 states "The accepting doctor is also responsible for coordinating with other disciplines for the patients total care." Is this sufficient to meet the multidisciplinary coordination requirements in the JCI ACC chapter?',
        'expected_standard': 'ACC.3',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.14 states "The accepting doctor must ensure there is a bed available in the ward to transfer the patient into and be responsible for reviewing the patient when he/she arrives. If there is a need to admit to an ICU or to overflow into other wards, the accepting doctor has to liaise with the ICU doctor or the overflow ward doctor to arrange for the bed." How does this compare to the patient flow management requirements in the JCI ACC chapter?',
        'expected_standard': 'ACC.2',
        'expected_chapter':  'ACC',
        'task_type':         'comparison',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.15 states "The outpatient facilities should not be used as a location for doctors to assess inpatients from other hospitals to determine suitability and need for transfer." What is the basis for this requirement within the JCI ACC chapter?',
        'expected_standard': 'ACC.1',
        'expected_chapter':  'ACC',
        'task_type':         'clarification',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.3 states "The team receiving the patient should ensure that all pertinent information communicated is transcribed and recorded in the patients medical record and/or relevant document(s)." What additional documentation elements does the JCI ACC chapter require to be captured in the medical records of transferred patients?',
        'expected_standard': 'ACC.5.1',
        'expected_chapter':  'ACC',
        'task_type':         'clarification',
    },   
    {
        'query':             'The SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004) requires the accepting doctor to assess the suitability of a patient for transfer before agreeing to proceed. Does this align with the requirements of JCI Standard ACC.5?',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'The SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004) requires a hospital text-based messaging platform (e.g. TigerConnect) to be set up upon confirmation of inter-hospital transfer for communication of patient condition and management. Does this satisfy JCI Standard ACC.5.1 written clinical summary requirement?',
        'expected_standard': 'ACC.5.1',
        'expected_chapter':  'ACC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'The SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004) addresses the benefits-versus-risks assessment for transfer decisions. How does this approach compare to the criteria-based transfer framework required under JCI Standard ACC.5?',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'comparison',
    },
    {
        'query':             'The SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004) references the Ministry of Health Professional Circular No. MH 53:08/4 vol 6 as a non-SGH reference document. Is this approach to referencing external regulatory documents sufficient under JCI Standard FMS.1?',
        'expected_standard': 'FMS.1',
        'expected_chapter':  'FMS',
        'task_type':         'compliance_check',
    },
    {
        'query':             'The SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004) addresses staffing requirements for patient transfers, specifying doctor grades for critically ill patients. How does this compare to the staffing qualification requirements under JCI Standard ACC.5, Measurable Element 3?',
        'expected_standard': 'ACC.5',
        'expected_chapter':  'ACC',
        'task_type':         'comparison',
    },
]

print(f'ACC gold standard: {len(ACC_gold_standard)} queries loaded.')

Policy gold standard: 15 queries loaded.
Section 6 complete.


In [ ]:
PCC_gold_standard = [
    {
        'query':             'In SGH\'s Intra-Hospital and Inter-Hospital Transfer policy (MB-ACC-004), clause 5.1.8 states that "the doctor and nurse taking care of the patient shall inform the patient and the relatives as appropriate prior to the transfer." Does this meet the requirements of JCI standard PCC.3.1 regarding the patient\'s right to participate in care decisions?',
        'expected_standard': 'PCC.3.1',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'SGH\'s transfer policy (MB-ACC-004), clause 3.2 states that "if transfer is at patient\'s request, the patient should be advised on the risk and benefits of the transfer by both the referring and accepting doctors." Does this satisfy JCI standard PCC.4.3\'s requirements for informed decision-making?',
        'expected_standard': 'PCC.4.3',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In SGH\'s transfer policy (MB-ACC-004), clause 5.2.6 requires that "the referring doctor and nurse taking care of the patient shall inform the patient and the relatives as appropriate prior to the transfer." Does this meet JCI standard PCC.4.4\'s requirements for surrogate consent when a patient cannot make decisions?',
        'expected_standard': 'PCC.4.4',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'SGH\'s transfer policy (MB-ACC-004), clause 5.1.4 states that "the referring team should summarize the case history of patient and document it in SCM before transferring the patient to the other discipline. Information relayed includes health status of patient, summary of care provided, patient\'s progress and the reason for transfer." Does this meet JCI standard PCC.5.1\'s requirements for documenting patient educational needs?',
        'expected_standard': 'PCC.5.1',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In SGH\'s transfer policy (MB-ACC-004), clause 5.2.7 states that "the procedures and other interventions provided should be written in the patient\'s discharge summary and given to the accepting staff taking over the care of the patient." Does this satisfy JCI standard PCC.5.2\'s requirement for reinforcing verbal information with written materials for patients and families?',
        'expected_standard': 'PCC.5.2',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'SGH\'s transfer policy (MB-ACC-004), clause 3.10 states that "if transfer is at patient\'s request, the patient should be advised on the risk and benefits of the transfer by both the referring and accepting doctors." How does this compare to the broader patient consent requirements outlined in the Patient Consent Process section of the JCI PCC chapter?',
        'expected_standard': 'PCC.4',
        'expected_chapter':  'PCC',
        'task_type':         'comparison',
    },
    {
        'query':             'In SGH\'s transfer policy (MB-ACC-004), clause 5.1.9 requires that the "transferring ward RN in charge contacts receiving ward RN in charge to provide information on patient\'s condition, supporting equipment required and management plan." How does this compare to the Patient and Family Education requirements in the JCI PCC chapter?',
        'expected_standard': 'PCC.5',
        'expected_chapter':  'PCC',
        'task_type':         'comparison',
    },
    {
        'query':             'SGH\'s transfer policy (MB-ACC-004), clause 5.2.13 states that "the accepting doctor is also responsible for coordinating with other disciplines for the patient\'s total care." Does this align with the patient complaint and conflict resolution requirements in the Patient Experience section of the JCI PCC chapter?',
        'expected_standard': 'PCC.3',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In SGH\'s transfer policy (MB-ACC-004), clause 5.2.14 states that "the accepting doctor must ensure there is a bed available in the ward to transfer the patient into and be responsible for reviewing the patient when he/she arrives." Does this address the patient\'s right to participate in care decisions as outlined in the Patient Consent Process section of the JCI PCC chapter?',
        'expected_standard': 'PCC.4',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'SGH\'s transfer policy (MB-ACC-004), clause 5.2.15 states that "the outpatient facilities should not be used as a location for doctors to assess inpatients from other hospitals to determine suitability and need for transfer." How does this relate to the Organ and Tissue Donation Information section of the JCI PCC chapter in terms of ensuring appropriate care settings?',
        'expected_standard': 'PCC.6',
        'expected_chapter':  'PCC',
        'task_type':         'clarification',
    },
    {
        'query':             'Does SGH\'s Intra-Hospital and Inter-Hospital Transfer policy (MB-ACC-004) address the need to assess and accommodate language and literacy barriers when communicating with patients and families about their transfer, as required by JCI standard PCC.5.2?',
        'expected_standard': 'PCC.5.2',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Does SGH\'s Intra-Hospital and Inter-Hospital Transfer policy (MB-ACC-004) ensure that patients being transferred to step-down units receive education about continuing their care, as required by JCI standard PCC.5.1?',
        'expected_standard': 'PCC.5.1',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Does SGH\'s Intra-Hospital and Inter-Hospital Transfer policy (MB-ACC-004) establish a process for patients to raise complaints or conflicts about transfer decisions, as required by JCI standard PCC.3.1?',
        'expected_standard': 'PCC.3.1',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'Does SGH\'s Intra-Hospital and Inter-Hospital Transfer policy (MB-ACC-004) require that patients know the identity of the healthcare practitioners responsible for their care during and after the transfer, as required by JCI standard PCC.4.3?',
        'expected_standard': 'PCC.4.3',
        'expected_chapter':  'PCC',
        'task_type':         'compliance_check',
    },
    {
        'query':             'How does SGH\'s Intra-Hospital and Inter-Hospital Transfer policy (MB-ACC-004) compare to JCI standard PCC.5\'s requirement for a collaborative, structured education programme when informing patients and families about the transfer?',
        'expected_standard': 'PCC.5',
        'expected_chapter':  'PCC',
        'task_type':         'comparison',
    },
]
print(f'PCC gold standard: {len(PCC_gold_standard)} queries loaded.')

In [ ]:
COP_gold_standard = [
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 1.1 states "To provide guidelines for healthcare team to comply during intra-hospital and inter-hospital transfers of patients." Does this purpose statement adequately reflect the JCI COP requirement for individualised patient care planning during transfers?',
        'expected_standard': 'COP.2',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 3.6 states "The referring doctor should be familiar with the patient, to assist in answering the questions asked by the accepting team. For intra-hospital transfer, critically ill cases, communication of the details of the transfer and the condition of the patient shall be done between doctors of at least Senior Resident grade from both the referring and accepting team." How does this compare with the JCI COP standard on qualified staff delivering care?',
        'expected_standard': 'COP.1',
        'expected_chapter':  'COP',
        'task_type':         'comparison',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.1.4 states "The referring team should summarize the case history of patient and document it in SCM before transferring the patient to the other discipline. Information relayed includes health status of patient, summary of care provided, patients progress and the reason for transfer." Does this comply with JCI COP standards on the content of patient care documentation?',
        'expected_standard': 'COP.2.1',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 3.5 states "Attempts should be made to stabilize the condition of the patient prior to the transfer. Vital signs should be assessed prior to transfer." Does this align with JCI COP standards on patient assessment before care transitions?',
        'expected_standard': 'COP.1',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clauses 5.1.8 and 5.2.6 state that "The doctor and nurse taking care of the patient shall inform the patient and the relatives as appropriate prior to the transfer." How does this compare with JCI COP standards on patient and family involvement in care?',
        'expected_standard': 'COP.2',
        'expected_chapter':  'COP',
        'task_type':         'comparison',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.1.10 states "The team from the accepting ward shall be available to receive and verify patients identity, receive patients report and clarify any doubts when the patient arrives in the accepting ward." Does this meet JCI COP standards on patient reassessment upon care transitions?',
        'expected_standard': 'COP.2',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.1.3 states "In the case of a patient being cared for by multiple disciplines, the doctors involved should communicate with each other to decide who shall be the primary physician organising the transfer." Does this satisfy JCI COP standards on multidisciplinary care coordination?',
        'expected_standard': 'COP.2.1',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.2 states that upon confirmation of transfer, a hospital text-based messaging platform (e.g. TigerConnect) shall be immediately set up by the referring hospital to facilitate communication of the patients condition and management. Does this satisfy JCI COP standards on care coordination and communication documentation?',
        'expected_standard': 'COP.2',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.7 states "The procedures and other interventions provided should be written in the patients discharge summary and given to the accepting staff taking over the care of the patient." Does this comply with JCI COP standards on care documentation at transitions?',
        'expected_standard': 'COP.2.1',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.12 states "Critically ill patients should have their vital signs monitored throughout the entire journey to another hospital." How does this compare with JCI COP standards on patient monitoring during care transitions?',
        'expected_standard': 'COP.1',
        'expected_chapter':  'COP',
        'task_type':         'comparison',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.13 states "The staff accompanying the patient during transfer shall hand over the information of care and management of the patient to the accepting institution with relevant documents. This is to be documented in the Handover Note (Appendix 2 NU-OPS-426)." Does this meet JCI COP standards on the transfer of care information between providers?',
        'expected_standard': 'COP.2',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'The SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004) does not contain any reference to pain assessment or management during the transfer process. What does JCI COP Standard COP.6 require in relation to pain management during patient transfers, and does the absence of such provisions represent a compliance gap?',
        'expected_standard': 'COP.6',
        'expected_chapter':  'COP',
        'task_type':         'clarification',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.13 states "The accepting doctor is also responsible for coordinating with other disciplines for the patients total care." Is this sufficient to meet the integrated multidisciplinary care planning requirements under JCI COP standards?',
        'expected_standard': 'COP.2.1',
        'expected_chapter':  'COP',
        'task_type':         'compliance_check',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 3.4 states "If a patient wishes to be transferred on financial grounds, Standards 3.2 and 3.3 shall be met. For patients transferring from a private hospital to a public hospital, the prevailing financial rules apply." How does this approach to financially motivated transfers align with JCI COP standards on equitable patient care?',
        'expected_standard': 'COP.1',
        'expected_chapter':  'COP',
        'task_type':         'comparison',
    },
    {
        'query':             'In the SGH Intra-Hospital and Inter-Hospital Transfer Policy (MB-ACC-004), Clause 5.2.16 states "Inpatients from other hospitals shall not be transferred to or received by the Department of Emergency Medicine." What clarification does JCI COP standard COP.1 provide on directing patients to appropriate care settings, and does this clause require additional justification to be compliant?',
        'expected_standard': 'COP.1',
        'expected_chapter':  'COP',
        'task_type':         'clarification',
    },
]
print(f'COP gold standard: {len(COP_gold_standard)} queries loaded.')
print('Section 6 complete.')

---
## Section 7 — Validation Loop (Retrieval Only — No LLM)
For each query:
1. Parse → routing decision (with `all_standards` and `all_chapters`)
2. `route_and_retrieve()` → three-tier routing
3. Qwen reranker → scores chunks
4. Record top-1 and top-3 matches vs expected

Results saved to `PARSER_VALIDATION.csv`.

In [ ]:
# ── Choose which gold standards to run ────────────────────────
gold_standards_to_run = {
    'ACC': ACC_gold_standard,
    'PCC': PCC_gold_standard,
    'COP': COP_gold_standard,
}

# ── Loop over each gold standard set ──────────────────────────
for chapter_name, policy_gold_standard in gold_standards_to_run.items():

    print(f'\n{"="*60}')
    print(f'Running: {chapter_name} ({len(policy_gold_standard)} queries)')
    print(f'{"="*60}')

    rows_top1 = []
    rows_top3 = []
    rows_top5 = []
    rows_top7 = []

    for i, item in enumerate(policy_gold_standard, 1):
        query        = item['query']
        expected_std = item['expected_standard'].strip()
        expected_ch  = item['expected_chapter']

        print(f'[{i:02d}/{len(policy_gold_standard)}] Expected: {expected_std}')

        parsed = parse_query(query, print_json=False)
        retrieved = route_and_retrieve(query, parsed, jci_dbs, chapter_chunks)
        qwen_ranked = rerank_qwen(query, retrieved)

        exact_std_codes = set(parsed.all_standards)
        exact_chunks = sorted([(doc, score) for doc, score in qwen_ranked if doc.metadata.get('standard_code') in exact_std_codes], key=lambda x: x[1], reverse=True)
        other_chunks = sorted([(doc, score) for doc, score in qwen_ranked if doc.metadata.get('standard_code') not in exact_std_codes], key=lambda x: x[1], reverse=True)
        guaranteed_ranked = exact_chunks + other_chunks

        top1_doc,  top1_score  = guaranteed_ranked[0]
        top3       = guaranteed_ranked[:3]
        top5       = guaranteed_ranked[:5]
        top7       = guaranteed_ranked[:7]

        top1_std    = top1_doc.metadata.get('standard_code')
        top3_stds   = [doc.metadata.get('standard_code') for doc, _ in top3]
        top5_stds   = [doc.metadata.get('standard_code') for doc, _ in top5]
        top7_stds   = [doc.metadata.get('standard_code') for doc, _ in top7]

        top3_scores = [round(s, 4) for _, s in top3]
        top5_scores = [round(s, 4) for _, s in top5]
        top7_scores = [round(s, 4) for _, s in top7]

        top1_match = top1_std == expected_std
        top3_match = expected_std in str(top3_stds)
        top5_match = expected_std in str(top5_stds)
        top7_match = expected_std in str(top7_stds)

        shared = {
            'query':                     query,
            'expected_standard':         expected_std,
            'expected_chapter':          expected_ch,
            'gold_task_type':            item['task_type'],
            'parsed_task_type':          parsed.task_type,
            'parsed_chapter':            parsed.reference_chapter,
            'parsed_all_standards':      str(parsed.all_standards),
            'parsed_all_chapters':       str(parsed.all_chapters),
            'parsed_measurable_element': parsed.measurable_element,
            'routing_source':            parsed.routing_source,
            'retrieval_scope':           parsed.retrieval_scope,
        }

        rows_top1.append({**shared, 'retrieved_standard': top1_std, 'retrieved_score': top1_score, 'retrieved_chunks': top1_doc.page_content.strip(), 'retrieved_chunks_json': json.dumps([{'standard_code': top1_doc.metadata.get('standard_code'), 'chapter': top1_doc.metadata.get('chapter'), 'score': round(top1_score, 4), 'text': top1_doc.page_content.strip()}]), 'top1_match': top1_match})
        rows_top3.append({**shared, 'retrieved_standards': str(top3_stds), 'retrieved_scores': str(top3_scores), 'retrieved_chunks': '\n\n---\n\n'.join(doc.page_content.strip() for doc, _ in top3), 'retrieved_chunks_json': json.dumps([{'standard_code': doc.metadata.get('standard_code'), 'chapter': doc.metadata.get('chapter'), 'score': round(score, 4), 'text': doc.page_content.strip()} for doc, score in top3]), 'top1_match': top1_match, 'top3_match': top3_match})
        rows_top5.append({**shared, 'retrieved_standards': str(top5_stds), 'retrieved_scores': str(top5_scores), 'retrieved_chunks': '\n\n---\n\n'.join(doc.page_content.strip() for doc, _ in top5), 'retrieved_chunks_json': json.dumps([{'standard_code': doc.metadata.get('standard_code'), 'chapter': doc.metadata.get('chapter'), 'score': round(score, 4), 'text': doc.page_content.strip()} for doc, score in top5]), 'top1_match': top1_match, 'top5_match': top5_match})
        rows_top7.append({**shared, 'retrieved_standards': str(top7_stds), 'retrieved_scores': str(top7_scores), 'retrieved_chunks': '\n\n---\n\n'.join(doc.page_content.strip() for doc, _ in top7), 'retrieved_chunks_json': json.dumps([{'standard_code': doc.metadata.get('standard_code'), 'chapter': doc.metadata.get('chapter'), 'score': round(score, 4), 'text': doc.page_content.strip()} for doc, score in top7]), 'top1_match': top1_match, 'top7_match': top7_match})

        status = '✓' if top1_match else '✗'
        print(f'  [{status}] Got: {top1_std} (score: {top1_score:.4f}) | tier: {parsed.routing_source} | all_standards: {parsed.all_standards}')

    # ── Save files for this chapter set ───────────────────────
    pd.DataFrame(rows_top1).to_csv(f'VALIDATION_TOP1_{chapter_name}.csv', index=False)
    pd.DataFrame(rows_top3).to_csv(f'VALIDATION_TOP3_{chapter_name}.csv', index=False)
    pd.DataFrame(rows_top5).to_csv(f'VALIDATION_TOP5_{chapter_name}.csv', index=False)
    pd.DataFrame(rows_top7).to_csv(f'VALIDATION_TOP7_{chapter_name}.csv', index=False)

    all_chunks_json = []
    for row1, row3, row5, row7 in zip(rows_top1, rows_top3, rows_top5, rows_top7):
        all_chunks_json.append({
            'query':                row1['query'],
            'expected_standard':    row1['expected_standard'],
            'parsed_all_standards': row1['parsed_all_standards'],
            'routing_source':       row1['routing_source'],
            'gold_task_type':       row1['gold_task_type'],
            'top1_chunks':          json.loads(row1['retrieved_chunks_json']),
            'top3_chunks':          json.loads(row3['retrieved_chunks_json']),
            'top5_chunks':          json.loads(row5['retrieved_chunks_json']),
            'top7_chunks':          json.loads(row7['retrieved_chunks_json']),
        })

    with open(f'VALIDATION_CHUNKS_{chapter_name}.json', 'w') as f:
        json.dump(all_chunks_json, f, indent=2)

    val_df = pd.DataFrame(rows_top1)
    print(f'\nAccuracy summary for {chapter_name}:')
    print(f'  Top-1 : {pd.DataFrame(rows_top1)["top1_match"].mean()*100:.1f}%')
    print(f'  Top-3 : {pd.DataFrame(rows_top3)["top3_match"].mean()*100:.1f}%')
    print(f'  Top-5 : {pd.DataFrame(rows_top5)["top5_match"].mean()*100:.1f}%')
    print(f'  Top-7 : {pd.DataFrame(rows_top7)["top7_match"].mean()*100:.1f}%')
    print(f'Saved: VALIDATION_TOP1/3/5/7_{chapter_name}.csv')
    print(f'Saved: VALIDATION_CHUNKS_{chapter_name}.json')

print('Section 7 complete.')

[01/9] Expected: ACC.3
  Tier 1 direct lookup: 6 chunks for ['ACC.3']
  Tier 1 returned 6 chunks — supplementing with chapter search
  [✓] Got: ACC.3 (score: 1.0000) | tier: explicit | all_standards: ['ACC.3']
[02/9] Expected: ACC.5.1
  Tier 1 direct lookup: 3 chunks for ['ACC.5.1']
  Tier 1 returned 3 chunks — supplementing with chapter search
  [✓] Got: ACC.5.1 (score: 1.0000) | tier: explicit | all_standards: ['ACC.5.1']
[03/9] Expected: ACC.3
  Tier 1 direct lookup: 6 chunks for ['ACC.3']
  Tier 1 returned 6 chunks — supplementing with chapter search
  [✗] Got: ACC.5.1 (score: 0.9922) | tier: explicit | all_standards: ['ACC.3']
[04/9] Expected: ACC.5
  Tier 1 direct lookup: 6 chunks for ['ACC.5']
  Tier 1 returned 6 chunks — supplementing with chapter search
  [✓] Got: ACC.5 (score: 0.9922) | tier: explicit | all_standards: ['ACC.5']
[05/9] Expected: ACC.5
  Tier 1 direct lookup: 6 chunks for ['ACC.5']
  Tier 1 returned 6 chunks — supplementing with chapter search
  [✗] Got: ACC.1 

---
## Section 8 — Compliance Analysis (Mistral)


Generates three answers per query using top-3, top-5, and top-7 chunks.
This lets you compare which context window gives the best compliance verdict.

Results saved to `COMPLIANCE_ANALYSIS.csv`.

In [ ]:
#Baseline prompt
COMPLIANCE_SYSTEM_PROMPT = """
You are a Policy Review and Healthcare Compliance Expert (RAG Mode) with specialized expertise in:
MOH JCI ESS Hospital Standards (7th Edition) document.

You operate in a Retrieval‑Augmented Generation (RAG) environment. Retrieved documents are your primary and authoritative source of truth.

You will be given:
- A user query 
    [The query may include Hospital policy excerpts]
- A task type:
    -> compliance_check
    -> comparison
    -> clarification
    If absent, treat as general_qa
- One or more retrieved document chunks of the JCI ESS Standards (7th Edition) document.

You must assume:
Retrieved chunks may be partial, fragmented, overlapping, or incomplete
Not all necessary information may be present

CORE RAG PRINCIPLES (NON‑NEGOTIABLE)
1. Retrieval‑First, Source‑Bound Reasoning
    - Assess and analyze information only using the retrieved chunks and content explicitly provided by the user.
    - Treat retrieved documents as authoritative.
    - Do not rely on memory, training data, or assumed knowledge of the standards beyond the retrieved text.
2. No Hallucination or Gap‑Filling
    - Do not invent:
        -> Clause numbers
        -> Section titles
        -> Measurable elements
        
Regulatory requirements:
- Do not fill gaps with assumed “typical JCI expectations.”
- If a requirement is not present in the retrieved chunks, treat it as not evidenced.

TASK TYPE ROUTING LOGIC (MANDATORY)
A. compliance_check
    - When the task type is Compliance Check:
        -> Assess alignment between the policy content and the retrieved standards only.
    - Classify findings implicitly as:
        -> Aligned
        -> Partially aligned
        -> Not evidenced / cannot be assessed
    - Identify:
        -> Missing or insufficient elements relative to retrieved content
        -> Weak documentation, accountability, monitoring, or review mechanisms
    - Each identified gap must be:
        -> Anchored to specific retrieved text
        -> Explained in terms of risk or compliance impact
        -> If retrieved chunks lack coverage to assess compliance, explicitly state: “Compliance cannot be fully assessed based on the retrieved references.”

B. comparison
    - When the task type is Comparison:
        -> Compare policy content versus retrieved standards only.
    - Highlight:
        -> Differences in scope, specificity, or wording
        -> Requirements present in retrieved standards but absent in the policy
    - Do NOT infer intent or expand beyond what the retrieved text explicitly states.

C. clarification
    - When the task type is Clarification:
        -> Explain or interpret retrieved clauses, intents, or terminology.
    - Stay strictly within:
        -> The language
        -> The intent
        -> The context of the retrieved text
    - Clearly STATE when an explanation IS an interpretation, NOT an explicit requirement.

D. general_qa (No Task Type Provided)
    - Answer directly using retrieved documents as the source of truth.
    - If multiple retrieved chunks conflict, acknowledge the inconsistency rather than resolving it independently.

MANDATORY UNCERTAINTY HANDLING
You must explicitly signal uncertainty when applicable using direct and transparent language such as:
    - “The retrieved standards do not specify…”
    - “Based on the provided excerpts, this requirement is not explicitly stated…”
    - “The available references are insufficient to determine…”
NEVER mask uncertainty with confident or authoritative claims.

EVIDENCE ANCHORING RULES
Reference standards using:
    - Chapter names
    - Standard titles
    - Intent statements
    - Descriptive identifiers (if clause numbers are not available)
If exact identifiers are missing in the retrieved chunks:
    - Acknowledge this explicitly
Do NOT fabricate precision

OUTPUT FORMAT AND STYLE CONSTRAINTS
Respond in paragraph form
Do not use tables, bullet points, emojis, or full policy rewrites unless explicitly requested
Maintain a formal, professional, audit‑ready tone
Do not restate large portions of retrieved text verbatim unless necessary for clarity


PROHIBITED BEHAVIORS
You must NOT:
    - Assume compliance
    - Broaden scope beyond retrieved content
    - Apply “best practice” labels unless supported by retrieved documents
    - Provide regulatory assurances or legal conclusions
    - Rewrite entire policies unless explicitly requested


ANALYTICAL QUALITY EXPECTATIONS
Your response should demonstrate:
    - Disciplined RAG use with clear linkage to retrieved content
    - Accurate interpretation without overreach
    - Strong understanding of healthcare governance and quality principles
Clear differentiation between:
    - What is explicitly required
    - What is interpreted
    - What is unknown or missing
"""

In [ ]:
def chunks_to_documents(chunk_list: list) -> list:
    return [
        Document(
            page_content=chunk['text'],
            metadata={
                'standard_code': chunk['standard_code'],
                'chapter':       chunk['chapter'],
            }
        )
        for chunk in chunk_list
    ]


def build_compliance_prompt(query: str, chunks, task_type: str = 'general_qa') -> list:
    if isinstance(chunks, Document):
        chunks = [chunks]
    context_parts = []
    for idx, chunk in enumerate(chunks, 1):
        context_parts.append(
            f'[JCI Chunk {idx} | Standard: {chunk.metadata.get("standard_code", "?")} | '
            f'Chapter: {chunk.metadata.get("chapter", "?")}]\n'
            f'{chunk.page_content.strip()}'
        )
    context_str = '\n\n'.join(context_parts)
    user_content = (
        f'COMPLIANCE QUESTION:\n{query}\n\n'
        f'TASK TYPE: {task_type}\n\n'
        f'RELEVANT JCI STANDARDS:\n{context_str}\n\n'
        f'Assessment (cite the JCI standard code where relevant):'
    )
    return [
        {'role': 'system', 'content': COMPLIANCE_SYSTEM_PROMPT},
        {'role': 'user',   'content': user_content},
    ]


def generate_mistral_answer(messages: list) -> str:
    output = mistral_pipeline(messages, max_new_tokens=LLM_MAX_TOKENS)
    return output[0]['generated_text'][-1]['content'].strip()


# ── Load Mistral ──────────────────────────────────────────────
print(f'Loading LLM: {LLM_MODEL_ID}')
mistral_pipeline = transformers.pipeline(
    'text-generation', model=LLM_MODEL_ID,
    model_kwargs={'torch_dtype': torch.bfloat16},
    device_map='auto',
)
print(f'LLM loaded: {LLM_MODEL_ID}')

# ── Chunk loading now handled inside the generation loop ──────
print('Section 8 setup complete. Run the generation loop cell next.')

Loading LLM: mistralai/Mistral-7B-Instruct-v0.3


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 192.60it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


LLM loaded: mistralai/Mistral-7B-Instruct-v0.3
Loaded 9 entries from VALIDATION_CHUNKS.json


In [ ]:
# ── Choose which sets to generate answers for ─────────────────
gold_standards_to_generate = {
    'ACC': 'VALIDATION_CHUNKS_ACC.json'
    'PCC': 'VALIDATION_CHUNKS_PCC.json',
    'COP': 'VALIDATION_CHUNKS_COP.json',
}

# ── Loop over each set ────────────────────────────────────────
for chapter_name, chunks_file in gold_standards_to_generate.items():

    print(f'\n{"="*60}')
    print(f'Generating answers: {chapter_name}')
    print(f'{"="*60}')

    with open(chunks_file, 'r') as f:
        chunks_data = json.load(f)
    print(f'Loaded {len(chunks_data)} entries from {chunks_file}')

    compliance_rows = []

    for i, entry in enumerate(chunks_data, 1):
        query        = entry['query']
        expected_std = entry['expected_standard']

        print(f'[{i:02d}/{len(chunks_data)}] Generating: {expected_std}')

        top3_docs = chunks_to_documents(entry['top3_chunks'])
        top5_docs = chunks_to_documents(entry['top5_chunks'])
        top7_docs = chunks_to_documents(entry['top7_chunks'])

        top3_stds   = [c['standard_code'] for c in entry['top3_chunks']]
        top5_stds   = [c['standard_code'] for c in entry['top5_chunks']]
        top7_stds   = [c['standard_code'] for c in entry['top7_chunks']]

        top3_scores = [c['score'] for c in entry['top3_chunks']]
        top5_scores = [c['score'] for c in entry['top5_chunks']]
        top7_scores = [c['score'] for c in entry['top7_chunks']]

        print('  Generating top-3 answer...')
        answer_top3 = generate_mistral_answer(
            build_compliance_prompt(query, top3_docs, task_type=entry['gold_task_type'])
        )
        print('  Generating top-5 answer...')
        answer_top5 = generate_mistral_answer(
            build_compliance_prompt(query, top5_docs, task_type=entry['gold_task_type'])
        )
        print('  Generating top-7 answer...')
        answer_top7 = generate_mistral_answer(
            build_compliance_prompt(query, top7_docs, task_type=entry['gold_task_type'])
        )

        answers_json = json.dumps([
            {'depth': 'top3', 'standards': top3_stds, 'scores': top3_scores, 'answer': answer_top3},
            {'depth': 'top5', 'standards': top5_stds, 'scores': top5_scores, 'answer': answer_top5},
            {'depth': 'top7', 'standards': top7_stds, 'scores': top7_scores, 'answer': answer_top7},
        ])

        compliance_rows.append({
            'query':             query,
            'expected_standard': expected_std,
            'routing_source':    entry['routing_source'],
            'all_standards':     entry['parsed_all_standards'],
            'answers_json':      answers_json,
            'top3_standards':    str(top3_stds),
            'top3_scores':       str(top3_scores),
            'top3_chunks':       '\n\n---\n\n'.join(d.page_content.strip() for d in top3_docs),
            'answer_top3':       answer_top3,
            'top5_standards':    str(top5_stds),
            'top5_scores':       str(top5_scores),
            'top5_chunks':       '\n\n---\n\n'.join(d.page_content.strip() for d in top5_docs),
            'answer_top5':       answer_top5,
            'top7_standards':    str(top7_stds),
            'top7_scores':       str(top7_scores),
            'top7_chunks':       '\n\n---\n\n'.join(d.page_content.strip() for d in top7_docs),
            'answer_top7':       answer_top7,
        })

    # ── Save CSV and JSON for this chapter set ─────────────────
    compliance_df = pd.DataFrame(compliance_rows)
    csv_name  = f'COMPLIANCE_ANALYSIS_Baseline_{chapter_name}.csv'
    json_name = f'COMPLIANCE_ANSWERS_Baseline_{chapter_name}.json'

    compliance_df.to_csv(csv_name, index=False)

    answers_list = [
        {
            'query':             row['query'],
            'expected_standard': row['expected_standard'],
            'answers':           json.loads(row['answers_json']),
        }
        for row in compliance_rows
    ]
    with open(json_name, 'w') as f:
        json.dump(answers_list, f, indent=2)

    print(f'Saved: {csv_name} ({len(compliance_df)} rows)')
    print(f'Saved: {json_name} ({len(answers_list)} entries)')

print('Section 8 complete.')

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[01/9] Generating: ACC.3
  Generating top-3 answer...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Generating top-5 answer...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Generating top-7 answer...


... (42 outputs left out)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[09/9] Generating: ACC.1
  Generating top-3 answer...
  Generating top-5 answer...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Generating top-7 answer...
Saved: COMPLIANCE_ANALYSIS_BaselinePrompt_OldQueries.csv (9 rows)
Saved: COMPLIANCE_ANSWERS_BaselinePrompt_OldQueries.json (9 entries)
Section 8 complete.
